# Reducción de una matriz simétrica a forma tridiagonal mediante Householder

## Bloque 1. Inicialización del problema

En este primer bloque se prepara toda la información necesaria antes de comenzar el algoritmo de Householder.

En particular, se realizan las siguientes tareas:

- Se importan las librerías necesarias para trabajar con matrices y vectores.
- Se define una **matriz simétrica** de ejemplo sobre la cual se aplicará el algoritmo.
- Se almacena una copia de la matriz original para verificar el resultado al finalizar la ejecución.
- Se obtiene la dimensión de la matriz.
- Se implementa la función `householder_vector(x)`, cuya finalidad es construir el **vector de Householder** asociado a un vector $x$.

Este vector permitirá construir posteriormente la matriz de Householder

$$
H = I - 2vv^{T},
$$

la cual se utilizará para introducir ceros debajo y encima de la diagonal, transformando gradualmente la matriz original en una **matriz tridiagonal**.

Al finalizar este bloque únicamente se inicializan los datos del problema; la reducción comienza en el siguiente bloque.

In [1]:
import numpy as np

# ============================================================
# REDUCCIÓN DE HOUSEHOLDER A FORMA TRIDIAGONAL
# ============================================================
# Objetivo:
# Reducir una matriz simétrica A a una matriz tridiagonal T
# mediante transformaciones ortogonales de Householder.
#
# Al finalizar se verifica
#
#            T = Qᵀ A Q
#
# donde:
#
# Q : matriz ortogonal
# T : matriz tridiagonal
#
# ============================================================


# ============================================================
# MATRIZ DE ENTRADA
# ============================================================

# La matriz debe ser simétrica.

A = np.array([
    [ 4.,  1., -2.,  2.],
    [ 1.,  2.,  0.,  1.],
    [-2.,  0.,  3., -2.],
    [ 2.,  1., -2., -1.]
], dtype=float)

# Copia de la matriz original (para la verificación final)

A_original = A.copy()

# Dimensión de la matriz

n = A.shape[0]


# ============================================================
# FUNCIÓN:
# Construcción del reflector de Householder
# ============================================================

def householder_vector(x):
    """
    Calcula el vector de Householder asociado al vector x.

    Parámetros
    ----------
    x : ndarray
        Vector columna.

    Retorna
    -------
    v : ndarray
        Vector de Householder normalizado.
    """

    # --------------------------------------------------------
    # Norma euclidiana de x
    # --------------------------------------------------------

    alpha = np.linalg.norm(x)

    # --------------------------------------------------------
    # Si x es el vector nulo,
    # no existe reflector.
    # --------------------------------------------------------

    if alpha == 0:
        return x

    # --------------------------------------------------------
    # Construcción del vector canónico e1
    # --------------------------------------------------------

    e1 = np.zeros_like(x)

    e1[0] = 1.0

    # --------------------------------------------------------
    # Se evita cancelación numérica
    # --------------------------------------------------------

    if x[0] >= 0:
        v = x + alpha * e1
    else:
        v = x - alpha * e1

    # --------------------------------------------------------
    # Normalización
    # --------------------------------------------------------

    v = v / np.linalg.norm(v)

    return v


# ============================================================
# IMPRESIÓN DE LOS DATOS
# ============================================================

print("="*65)
print(" REDUCCIÓN DE HOUSEHOLDER A MATRIZ TRIDIAGONAL")
print("="*65)

print("\nMatriz inicial A\n")

print(A)

print("\nDimensión =", n)

 REDUCCIÓN DE HOUSEHOLDER A MATRIZ TRIDIAGONAL

Matriz inicial A

[[ 4.  1. -2.  2.]
 [ 1.  2.  0.  1.]
 [-2.  0.  3. -2.]
 [ 2.  1. -2. -1.]]

Dimensión = 4


# Bloque 2. Construcción de las transformaciones de Householder

En este bloque comienza la reducción de la matriz simétrica.

Para cada columna de la matriz se construye una **transformación de Householder**, cuyo objetivo es anular todos los elementos ubicados por debajo de la primera subdiagonal.

En cada iteración se realizan las siguientes operaciones:

- Se extrae la parte de la columna que debe transformarse.
- Se construye el vector de Householder correspondiente.
- Se forma la matriz de Householder $H_k$.
- Se aplica la transformación de similitud

$$
A \leftarrow H_kAH_k,
$$

la cual conserva los valores propios de la matriz.

Además, se acumulan todas las transformaciones ortogonales en una matriz \(Q\), que posteriormente permitirá verificar que

$$
T=Q^{T}AQ,
$$

donde $T$ será la matriz tridiagonal obtenida al finalizar el algoritmo.

In [2]:
# ============================================================
# BLOQUE 2
# Construcción de las transformaciones de Householder
# ============================================================

# ------------------------------------------------------------
# Matriz ortogonal acumulada
# ------------------------------------------------------------

Q = np.eye(n)                         # Inicialmente Q = I

# ------------------------------------------------------------
# Inicio del algoritmo
# ------------------------------------------------------------

for k in range(n - 2):

    print("\n" + "="*65)
    print(f"ITERACIÓN {k+1}")
    print("="*65)

    # --------------------------------------------------------
    # Extraer la parte de la columna que será transformada
    # --------------------------------------------------------

    x = A[k+1:, k]

    print("\nVector x\n")
    print(x)

    # --------------------------------------------------------
    # Calcular el vector de Householder
    # --------------------------------------------------------

    v = householder_vector(x)

    print("\nVector de Householder v\n")
    print(v)

    # --------------------------------------------------------
    # Construir la matriz de Householder reducida
    # --------------------------------------------------------

    Hk = np.eye(len(x))

    Hk = Hk - 2*np.outer(v, v)

    print("\nReflector reducido Hk\n")
    print(Hk)

    # --------------------------------------------------------
    # Construir el reflector completo
    # --------------------------------------------------------

    H = np.eye(n)

    H[k+1:, k+1:] = Hk

    print("\nMatriz de Householder H\n")
    print(H)

    # --------------------------------------------------------
    # Aplicar la transformación
    #
    # A <- H A H
    # --------------------------------------------------------

    A = H @ A @ H

    # --------------------------------------------------------
    # Acumular las transformaciones ortogonales
    # --------------------------------------------------------

    Q = Q @ H

    # --------------------------------------------------------
    # Mostrar la matriz transformada
    # --------------------------------------------------------

    print("\nMatriz transformada\n")

    print(np.round(A,6))


ITERACIÓN 1

Vector x

[ 1. -2.  2.]

Vector de Householder v

[ 0.81649658 -0.40824829  0.40824829]

Reflector reducido Hk

[[-0.33333333  0.66666667 -0.66666667]
 [ 0.66666667  0.66666667  0.33333333]
 [-0.66666667  0.33333333  0.66666667]]

Matriz de Householder H

[[ 1.          0.          0.          0.        ]
 [ 0.         -0.33333333  0.66666667 -0.66666667]
 [ 0.          0.66666667  0.66666667  0.33333333]
 [ 0.         -0.66666667  0.33333333  0.66666667]]

Matriz transformada

[[ 4.       -3.        0.       -0.      ]
 [-3.        3.333333  1.        1.333333]
 [ 0.        1.        1.666667 -1.333333]
 [-0.        1.333333 -1.333333 -1.      ]]

ITERACIÓN 2

Vector x

[1.         1.33333333]

Vector de Householder v

[0.89442719 0.4472136 ]

Reflector reducido Hk

[[-0.6 -0.8]
 [-0.8  0.6]]

Matriz de Householder H

[[ 1.   0.   0.   0. ]
 [ 0.   1.   0.   0. ]
 [ 0.   0.  -0.6 -0.8]
 [ 0.   0.  -0.8  0.6]]

Matriz transformada

[[ 4.       -3.        0.       -0.     

# Bloque 3. Verificación de la reducción a forma tridiagonal

Una vez finalizadas todas las transformaciones de Householder, es necesario comprobar que el algoritmo produjo el resultado esperado.

En este bloque se realizan las siguientes verificaciones:

- Se muestra la matriz tridiagonal obtenida.
- Se calcula la matriz

$$
Q^{T}AQ,
$$

empleando la matriz ortogonal acumulada durante todas las iteraciones.

- Se compara esta matriz con la matriz tridiagonal obtenida para comprobar que ambas coinciden.
- Finalmente, se calcula el error máximo entre ambas matrices. Si el algoritmo fue implementado correctamente, dicho error debe ser cercano a cero, considerando únicamente los errores de redondeo propios de la aritmética en punto flotante.

Estas verificaciones permiten confirmar que las transformaciones de Householder conservan los valores propios y producen correctamente una matriz tridiagonal semejante a la matriz original.

In [3]:
# ============================================================
# BLOQUE 3
# Verificación del algoritmo
# ============================================================

print("\n" + "="*65)
print("RESULTADOS FINALES")
print("="*65)

# ------------------------------------------------------------
# Matriz tridiagonal obtenida
# ------------------------------------------------------------

T = A.copy()                              # La matriz final corresponde a T

print("\nMatriz tridiagonal T\n")

print(np.round(T,6))

# ------------------------------------------------------------
# Verificación de la semejanza ortogonal
#
# T = Qᵀ A Q
# ------------------------------------------------------------

T_verificacion = Q.T @ A_original @ Q

print("\nQᵀ A Q\n")

print(np.round(T_verificacion,6))

# ------------------------------------------------------------
# Error máximo entre ambas matrices
# ------------------------------------------------------------

error = np.max(np.abs(T - T_verificacion))

print("\nError máximo")

print(error)

# ------------------------------------------------------------
# Comprobación de ortogonalidad
# ------------------------------------------------------------

print("\nQᵀ Q\n")

print(np.round(Q.T @ Q,6))

error_ortogonalidad = np.max(np.abs(Q.T @ Q - np.eye(n)))

print("\nError de ortogonalidad")

print(error_ortogonalidad)

# ------------------------------------------------------------
# Mensaje final
# ------------------------------------------------------------

if error < 1e-10:
    print("\nLa reducción a forma tridiagonal fue correcta.")
else:
    print("\nAdvertencia: existe un error mayor al esperado.")


RESULTADOS FINALES

Matriz tridiagonal T

[[ 4.       -3.        0.       -0.      ]
 [-3.        3.333333 -1.666667  0.      ]
 [ 0.       -1.666667 -1.32      0.906667]
 [-0.        0.        0.906667  1.986667]]

Qᵀ A Q

[[ 4.       -3.        0.       -0.      ]
 [-3.        3.333333 -1.666667  0.      ]
 [ 0.       -1.666667 -1.32      0.906667]
 [-0.        0.        0.906667  1.986667]]

Error máximo
4.440892098500626e-16

Qᵀ Q

[[ 1.  0.  0.  0.]
 [ 0.  1. -0.  0.]
 [ 0. -0.  1. -0.]
 [ 0.  0. -0.  1.]]

Error de ortogonalidad
6.661338147750939e-16

La reducción a forma tridiagonal fue correcta.


# Bloque 4. Interpretación de los resultados

En este bloque se realiza un análisis de la matriz tridiagonal obtenida mediante el algoritmo de Householder.

En particular, se verifica que:

- Los únicos elementos diferentes de cero se encuentran sobre la diagonal principal y las dos diagonales adyacentes.
- Los elementos alejados de estas diagonales son prácticamente cero (salvo errores numéricos de redondeo).
- La matriz transformada conserva los mismos valores propios que la matriz original, ya que las transformaciones de Householder son transformaciones ortogonales de similitud.

Finalmente, se presenta una interpretación del resultado obtenido y se explica por qué la reducción a forma tridiagonal constituye una etapa previa fundamental para algoritmos más eficientes, como el algoritmo QR para matrices simétricas.

In [4]:
# ============================================================
# BLOQUE 4
# Interpretación de los resultados
# ============================================================

print("\n" + "="*65)
print("INTERPRETACIÓN DE LOS RESULTADOS")
print("="*65)

# ------------------------------------------------------------
# Mostrar la matriz tridiagonal
# ------------------------------------------------------------

print("\nMatriz tridiagonal obtenida:\n")

print(np.round(T,6))

# ------------------------------------------------------------
# Identificar elementos fuera de la banda tridiagonal
# ------------------------------------------------------------

print("\nElementos fuera de la banda tridiagonal:\n")

for i in range(n):

    for j in range(n):

        if abs(i-j) > 1:

            print(f"T[{i},{j}] = {T[i,j]: .3e}")

# ------------------------------------------------------------
# Número de elementos prácticamente nulos
# ------------------------------------------------------------

contador = 0

for i in range(n):

    for j in range(n):

        if abs(i-j) > 1 and abs(T[i,j]) < 1e-10:

            contador += 1

print("\nNúmero de elementos prácticamente nulos")
print(contador)

# ------------------------------------------------------------
# Conclusión automática
# ------------------------------------------------------------

if contador == (n-1)*(n-2):

    print("\nLa matriz posee estructura tridiagonal.")
else:

    print("\nLa matriz aún presenta elementos fuera de la banda.")

# ------------------------------------------------------------
# Interpretación
# ------------------------------------------------------------

print("\nInterpretación")
print("-"*65)

print("• La matriz original fue transformada mediante")
print("  transformaciones ortogonales de Householder.")

print("\n• La matriz resultante conserva los mismos")
print("  valores propios que la matriz inicial.")

print("\n• Los elementos alejados de la diagonal principal")
print("  fueron eliminados, obteniéndose una matriz")
print("  tridiagonal.")

print("\n• Esta reducción disminuye el costo computacional")
print("  de algoritmos posteriores para el cálculo")
print("  de valores propios, como el algoritmo QR.")


INTERPRETACIÓN DE LOS RESULTADOS

Matriz tridiagonal obtenida:

[[ 4.       -3.        0.       -0.      ]
 [-3.        3.333333 -1.666667  0.      ]
 [ 0.       -1.666667 -1.32      0.906667]
 [-0.        0.        0.906667  1.986667]]

Elementos fuera de la banda tridiagonal:

T[0,2] =  1.332e-16
T[0,3] = -9.326e-16
T[1,3] =  0.000e+00
T[2,0] =  1.332e-16
T[3,0] = -9.326e-16
T[3,1] =  2.220e-16

Número de elementos prácticamente nulos
6

La matriz posee estructura tridiagonal.

Interpretación
-----------------------------------------------------------------
• La matriz original fue transformada mediante
  transformaciones ortogonales de Householder.

• La matriz resultante conserva los mismos
  valores propios que la matriz inicial.

• Los elementos alejados de la diagonal principal
  fueron eliminados, obteniéndose una matriz
  tridiagonal.

• Esta reducción disminuye el costo computacional
  de algoritmos posteriores para el cálculo
  de valores propios, como el algoritmo QR.


# Bloque 5. Conclusiones y análisis de los resultados

En este último bloque se presentan las conclusiones del algoritmo implementado.

A partir de los resultados obtenidos, se comparan la matriz original y la matriz tridiagonal, verificando que ambas poseen los mismos valores propios debido a que las transformaciones de Householder son transformaciones ortogonales de similitud.

Finalmente, se propone al estudiante analizar la estructura de la matriz tridiagonal obtenida y reflexionar sobre la importancia de esta reducción en el desarrollo de algoritmos eficientes para el cálculo de valores y vectores propios.

In [5]:
# ============================================================
# BLOQUE 5
# Conclusiones
# ============================================================

print("\n" + "="*65)
print("CONCLUSIONES")
print("="*65)

print("\nMatriz original A\n")
print(np.round(A_original,6))

print("\nMatriz tridiagonal T\n")
print(np.round(T,6))

# ------------------------------------------------------------
# Comparación de normas
# ------------------------------------------------------------

print("\nNorma de la matriz original")
print(np.linalg.norm(A_original))

print("\nNorma de la matriz tridiagonal")
print(np.linalg.norm(T))

# ------------------------------------------------------------
# Valores propios (solo para verificación)
# ------------------------------------------------------------
# Esta parte únicamente verifica el algoritmo.
# No forma parte de la implementación de Householder.

eigA = np.linalg.eigvalsh(A_original)

eigT = np.linalg.eigvalsh(T)

print("\nValores propios de la matriz original")

print(np.round(eigA,8))

print("\nValores propios de la matriz tridiagonal")

print(np.round(eigT,8))

print("\nDiferencia máxima entre valores propios")

print(np.max(np.abs(eigA-eigT)))

# ------------------------------------------------------------
# Conclusiones
# ------------------------------------------------------------

print("\nConclusiones")
print("-"*65)

print("1. El algoritmo redujo correctamente la matriz")
print("   simétrica a una matriz tridiagonal.")

print("\n2. La transformación utilizada es ortogonal,")

print("   por lo que conserva los valores propios.")

print("\n3. La matriz tridiagonal requiere menos")
print("   operaciones en algoritmos posteriores.")

print("\n4. La reducción a forma tridiagonal constituye")
print("   una etapa previa del algoritmo QR para")
print("   matrices simétricas.")

print("\n5. La implementación realizada sigue")
print("   el algoritmo clásico de Householder.")


CONCLUSIONES

Matriz original A

[[ 4.  1. -2.  2.]
 [ 1.  2.  0.  1.]
 [-2.  0.  3. -2.]
 [ 2.  1. -2. -1.]]

Matriz tridiagonal T

[[ 4.       -3.        0.       -0.      ]
 [-3.        3.333333 -1.666667  0.      ]
 [ 0.       -1.666667 -1.32      0.906667]
 [-0.        0.        0.906667  1.986667]]

Norma de la matriz original
7.615773105863909

Norma de la matriz tridiagonal
7.61577310586391

Valores propios de la matriz original
[-2.19751698  1.08436446  2.26853141  6.84462111]

Valores propios de la matriz tridiagonal
[-2.19751698  1.08436446  2.26853141  6.84462111]

Diferencia máxima entre valores propios
1.7763568394002505e-15

Conclusiones
-----------------------------------------------------------------
1. El algoritmo redujo correctamente la matriz
   simétrica a una matriz tridiagonal.

2. La transformación utilizada es ortogonal,
   por lo que conserva los valores propios.

3. La matriz tridiagonal requiere menos
   operaciones en algoritmos posteriores.

4. La reducc